# Seminar HCI and BCI in practice
## Session 6 Classification

***In this session the antagonistic finger movements are finally discriminated by means of three different classification algorithms.***


In [ ]:
import numpy as np
import os
import pickle
from scipy import stats
from nearly import nearly
from getBalancedTrainset import getBalancedTrainset
from train_bayes import train_bayes
from test_bayes import test_bayes
from train_lda import train_lda
from test_lda import test_lda
from classification_svm import classification_svm

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data')
print(f'Now you are located: {main_path}')

In [ ]:
ecog_file = os.path.join(data_path, 'raw/ecogStruct3.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Take a look into our data again
print("ecog contains")
for key, value in ecog.items():
    print(f"Key:{key}, Type:{type(value)}")

# Load epoch info
epoch_file = os.path.join(data_path, 'raw/epoch2.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print("\nepoch info:")
for key, value in epoch.items():
    print(f"Key:{key}, Type:{type(value)}")

In [ ]:
# Number of trials with finger movement
nTrials = np.array(ecog['periodogram']['periodogram']).shape[2]

# Frequency features (40-160 Hz based on Session 5 results)
freqBand = np.arange(40, 161)  # 40-160 Hz (inclusive)

# Find nearest frequency indices
freqIdx = np.unique(nearly(freqBand, ecog['periodogram']['centerFrequency']))

# Alternative:
# freqIdx = np.unique([np.argmin(np.abs(ecog['periodogram']['centerFrequency'] - f)) 
#                          for f in freqBand])
nFreq = len(freqIdx)

# Channel features (based on Session 5 results)
chan = np.array([17, 23, 29, 30, 39]) - 1  # Convert to 0-based indexing
nChan = len(chan)

# Prepare data for z-scoring (same as Session 4)
# Reshape to (nFreq, nChan*nTrials)
dat = np.array(ecog['periodogram']['periodogram'])[freqIdx, :, :][:, chan, :]
dat = dat.reshape(nFreq, nChan * nTrials, order='F')

# Z-score data along frequency axis
dat = stats.zscore(dat, axis=1) 

# Reshape data back to original structure with permutations
dat = dat.reshape(nFreq, nChan, nTrials, order='F')
dat = np.transpose(dat, (2, 1, 0)) 
dat = dat.reshape(nTrials, nFreq * nChan, order='F')

In [ ]:
## Create subsets for cross-validation
realClassLabels = np.array(epoch['label']) # Class labels
N = 10       # CV steps

selector = np.ceil((np.arange(1, len(realClassLabels)+1)) / (len(realClassLabels)/N))
selector = selector.astype(int)
selector = selector[np.random.permutation(len(realClassLabels))]

<h2 style="color: #FF0000; font-weight: bold;">TASK 1 (2 pt):</h2>

- What does the variable selector contain? 
- What is the purpose of cross-validation? Describe this method and its advantages/disadvantages.

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>


---

## Classification lda & bayes

In [ ]:
alg = 'bayes'     # 'bayes', 'lda'
banlance = True
predictedClassLabels = np.zeros(len(realClassLabels))

for cv in range(1, N+1):
    print(f'CV step #{cv}')
    testIdx = np.where(selector == cv)[0]
    trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)

    if banlance:
        labelIdx = getBalancedTrainset(realClassLabels[trainIdx])
        trainIdx = trainIdx[labelIdx]

    # Train Classifier
    curTrain = dat[trainIdx, :]
    curClassLabels = realClassLabels[trainIdx]
    
    if alg.lower() == 'bayes':
        R = train_bayes(curTrain, curClassLabels)
    elif alg.lower() == 'lda':
        R = train_lda(curTrain, curClassLabels)

    # Testing
    curTest = dat[testIdx, :]
    
    if alg.lower() == 'bayes':
        Res = test_bayes(R, curTest)
    elif alg.lower() == 'lda':
        Res = test_lda(R, curTest)
        
    predictedClassLabels[testIdx] = Res['prediction']

# accuracy rate
accuracy = np.sum(predictedClassLabels == realClassLabels) / len(realClassLabels)
print(f'{alg} Classification accuracy: {accuracy:.2%}')

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 (2 pt):</h2>

- What does it mean to balance datasets?
- Why is this sometimes done?
- Have a look at the `train_lda` and `train_bayes` functions. What kind of information do they store in the `dict` `R`?
- What is the difference between them? 

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (1 pt):</h2>

Have a look at the `test_lda` and `test_bayes` functions. What information is stored in `Res['prediction']`?

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>


---

## Classification SVM

In [ ]:
svm_results = classification_svm(dat, realClassLabels, selector, N, optimizeC=False)
predictedClassLabels = svm_results['predictedClassLabels']
accuracy_svm = svm_results['accuracy']
print(f"Final Accuracy: {accuracy_svm:.2%}")
predictedClassLabels.shape

In [ ]:
# Take a look into `svm_results`, try to understand the output from function `classification_svm`
for key, value in svm_results.items():
    print(f"Key:{key}, Type:{type(value)}")

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 (1 pt):</h2>

What is the cost parameter C and why is it iteratively optimized? Does it improve predictions you make?

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 5 (1 pt):</h2>

Compare the results of the 3 different (LDA, Bayes and SVM) algorithms. Which one produces the best results (highest accuracy)?

Then also change your selected features (channels/frequencies), always keeping in mind the results from last times t-values/relief algorithm. 

Which features lead to the highest accuracy? (Also keep in mind that the more features you use the longer the calculation time is, so try to reduce your number of features without this resulting in a lower accuracy.)

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 6:</h2>

If you have time left, you can try to use the data from the PCA (load `resultsPCA.pkl` - data saved in `xPCA` from Session 04) to perform the classification.

Use the information you got last week from the t-values to choose the best features.